## CS 598 PSL Fall 2026
### Project 2 Book Recommendations

Due Date: Monday, November 30 (11.59pm)

- Anirudh Eswara
- Ryan Lenea
- Mark Synowiec
- Robert Walker

This is a project, not a coding assignment. You will not be told which popularity formula to
use, which users to hold out, or how to break ties. You will be expected to make those choices,
document them, and defend them with the ideas from class: what a baseline already explains,
why neighborhood methods need a similarity and a fallback, how a held-out rating is not the
same thing as a held-out user, why a similarity matrix built on all ratings cannot be reused
for evaluation, and what RMSE can and cannot tell you about a top-10 list. Results in R and
Python need not match; that is expected. Using a canned IBCF implementation (for example
recommenderlab) as a stand-in for the function in part (B) is not acceptable. Sparse-matrix
libraries are fine.

## Project Description

The goal of this project is to make book recommendations. We will use the goodbooks-10k
data set. The current release of the CSV files can be found here:

[https://github.com/zygmuntz/goodbooks-10k](https://github.com/zygmuntz/goodbooks-10k)

Do not use the older Kaggle copy, which has duplicate ratings.

The data contain about 6 million ratings, on a 1–5 star scale, from 53,424 users for 10,000
books, together with book metadata (books.csv) and a to read list. Ratings in ratings.csv
are already sorted by time, but there is no per-user timestamp column. The book id used in
ratings.csv and to_read.csv is the 1–10,000 identifier in books.csv, not the Goodreads ID.

A 10,000 × 10,000 item–item similarity matrix is larger than you need for this project and will
be painful if computed carelessly. After loading the data, restrict the catalog to the N most rated
books in ratings.csv, with 2,500 ≤ N ≤ 4,000. State N and justify it. You may also
drop users with very few remaining ratings; if you do, say where you cut and why. All later
parts use this filtered rating matrix.

After loading, filtering, and pre-processing as needed, consider the following tasks.



In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data/goodbooks-10k")
N_BOOKS = 3000  # Assignment allows 2,500–4,000

ratings = pd.read_csv(DATA_DIR / "ratings.csv")
books = pd.read_csv(DATA_DIR / "books.csv")

# Select the N most-rated books; break ties by book_id for reproducibility.
rating_counts = (
    ratings.groupby("book_id")
    .size()
    .rename("rating_count")
    .reset_index()
    .sort_values(["rating_count", "book_id"], ascending=[False, True])
)

top_books = rating_counts.head(N_BOOKS)
top_book_ids = top_books["book_id"]

# Keep ratings and metadata for the selected catalog.
ratings_filtered = ratings[ratings["book_id"].isin(top_book_ids)].copy()
books_filtered = (
    books[books["book_id"].isin(top_book_ids)]
    .merge(top_books, on="book_id", how="inner")
    .sort_values(["rating_count", "book_id"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Selected books: {len(books_filtered):,}")
print(f"Filtered ratings: {len(ratings_filtered):,}")
print(f"Users represented: {ratings_filtered['user_id'].nunique():,}")
display(books_filtered[["book_id", "title", "authors", "rating_count"]].head())

In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(len(rating_counts["rating_count"])), rating_counts["rating_count"])

plt.xlabel("Ordered Books")
plt.ylabel("Ratings Per Book")
plt.title("Ratings Ordered by Count")

plt.show()

**Choice of 3,000 books: There is a significant drop off in ratings count in the first few hundred top books and afterwards rating count decreases nearly linearly. Expanding to 4,000 books would increase coverage from 76.1% to only 82%, but would roughly double storage. We selected the 3,000 most-rated books as a compromise between catalog coverage and computational cost.**

## (A) Top 10 "most popular" books
Recommend the top 10 “most popular” books. The term “most popular” is open
to interpretation. You must clearly state how you define the popularity of a book —
for example a raw rating count, a mean rating with a minimum-count cutoff, a Bayesian
average that shrinks toward the global mean, or something you defend. The recommendation
should display the top 10 titles (and authors), together with the numbers behind the
ranking. A ranking that is just “the 10 books with the most 5-star ratings” is acceptable
only if you say so and discuss what it misses.

In [ ]:
from IPython.display import display, Markdown

book_stats = (
    ratings_filtered.groupby("book_id")["rating"]
    .agg(rating_count="count", mean_rating="mean")
    .reset_index()
)
global_mean = ratings_filtered["rating"].mean()
book_metadata = books_filtered.drop(columns="rating_count")
display(Markdown(f"## Candidate Rankings"))

for m in [100, 500, 1000]:
    scored_books = book_stats.copy()
    scored_books["bayesian_score"] = (
        scored_books["rating_count"] * scored_books["mean_rating"]
        + m * global_mean
    ) / (scored_books["rating_count"] + m)

    top10 = (
        scored_books.merge(book_metadata, on="book_id", how="inner")
        .sort_values(["bayesian_score", "rating_count"], ascending=[False, False])
        .head(10)
    )

    display(Markdown(f"### Bayesian average, m = {m}"))
    display(top10[[
        "title", "authors", "rating_count", "mean_rating", "bayesian_score"
    ]].reset_index(drop=True).round(3))


## Final Ranking

In [ ]:
m = 500

scored_books = book_stats.copy()
scored_books["bayesian_score"] = (
    scored_books["rating_count"] * scored_books["mean_rating"]
    + m * global_mean
) / (scored_books["rating_count"] + m)

all_m500 = (
    scored_books.merge(book_metadata, on="book_id", how="inner")
    .sort_values(["bayesian_score", "rating_count"], ascending=[False, False])
)

top10_m500 = all_m500.head(10)

display(top10_m500[
    ["title", "authors", "rating_count", "mean_rating", "bayesian_score"]
].reset_index(drop=True).round(3))

**Analysis: we did not use a minimum rating cutoff as that seemed unnecessarily arbitrary. Instead, we used a Bayesian average to decrease the mean rating for books with fewer ratings which intuitively makes sense - a few five-star reviews don’t mean as much as many of them. We settled on m == 500 after seeing that The Complete Calvin and Hobbes were rated first with fewer than 500 reviews (which seemed unfair) when m == 100; Harry Potter books dominated the top 5 ratings with m == 1000 which seemed to be less useful. m == 500 seemed to be a sweet spot where books that had lots of ratings were elevated, but not so much so that the most-rated books entirely dominated.**

## (B) Item-Based Collaborative Filtering

Make recommendations with item-based collaborative filtering (IBCF) by writing
your own IBCF function, following the steps below. Use the full filtered rating matrix
here. Part (B) is how you inspect the method. It is not the matrix you will evaluate in (C).

1. Create the rating matrix R, with rows = users and columns = books in your filtered catalog. Normalize R by centering each row: subtract the row mean from each row. The matrix is sparse, so row means must be computed from the non-NA entries only.



In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

book_ids = books_filtered["book_id"].to_list()
book_index = {book_id: i for i, book_id in enumerate(book_ids)}
user_codes, unique_users = pd.factorize(ratings_filtered["user_id"])
item_codes = ratings_filtered["book_id"].map(book_index).to_numpy()

# Sparse user x book matrix of star ratings
R = csr_matrix(
    (ratings_filtered["rating"].to_numpy(dtype=float), (user_codes, item_codes)),
    shape=(len(unique_users), len(book_ids)),
)
R.sort_indices()
ratings_per_user = np.diff(R.indptr)

# Example: ratings [5, 4] have a mean of 4.5, so they become [+0.5, -0.5].
user_means = np.divide(
    np.asarray(R.sum(axis=1)).ravel(),
    ratings_per_user,
    out=np.zeros(R.shape[0]),
    where=ratings_per_user > 0,
)

R_centered = R.copy()
R_centered.data -= np.repeat(user_means, ratings_per_user)

# Print a 10 x 10 corner of R_centered (NA = unrated)
corner = R_centered[:10, :10].toarray()
corner[(R[:10, :10] == 0).toarray()] = np.nan
print(pd.DataFrame(corner).round(2))


2. Compute the cosine similarity for all pairs of books, using the centered matrix from (1). Ignore similarities computed from fewer than 3 co-raters. You may use the (1+cos)/2 transformation so that similarities lie in [0, 1]. You will have NAs when a pair of books has been co-rated by 0, 1, or 2 users, or when the denominator is zero (constant centered vectors).


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = (1 + cosine_similarity(R_centered.T)) / 2

# Count how many users rated each pair of books.
rated = R.copy()
rated.data[:] = 1
co_raters = (rated.T @ rated).toarray()

# Mark pairs with fewer than 3 co-raters, and zero-length book vectors, as missing.
book_norms = np.sqrt(np.asarray(R_centered.power(2).sum(axis=0)).ravel())
similarities[co_raters < 3] = np.nan
similarities[:, book_norms == 0] = np.nan
similarities[book_norms == 0, :] = np.nan
np.fill_diagonal(similarities, np.nan)

print(pd.DataFrame(similarities[:10, :10]).round(2))

3. For the similarity matrix in (2), keep only the top 30 non-NA similarities in each row and set the rest to NA. Display the pairwise similarity values from this truncated matrix for the following books (match titles to books.csv carefully; several Harry Potter and Hunger Games volumes exist):

- “The Hunger Games (The Hunger Games, #1)”
- “Harry Potter and the Sorcerer’s Stone (Harry Potter, #1)”
- “Pride and Prejudice”
- “The Iliad”
- “The Odyssey”

  Write two or three sentences on whether the neighbors look like a content-based genre match, a popularity match, or something less obvious.

In [ ]:
k = 30
top_neighbors = np.full_like(similarities, np.nan)

for i, row in enumerate(similarities):
    candidates = np.flatnonzero(np.isfinite(row))
    keep = candidates[np.argsort(row[candidates])[-k:]]
    top_neighbors[i, keep] = row[keep]

similarity_df = pd.DataFrame(top_neighbors, index=book_ids, columns=book_ids)

target_titles = [
    "The Hunger Games (The Hunger Games, #1)",
    "Harry Potter and the Sorcerer's Stone (Harry Potter, #1)",
    "Pride and Prejudice",
    "The Iliad",
    "The Odyssey",
]

targets = books_filtered[books_filtered["title"].isin(target_titles)]
ids = targets["book_id"].tolist()

pairwise = similarity_df.loc[ids, ids]
pairwise.index = targets["title"].tolist()
pairwise.columns = targets["title"].tolist()

display(pairwise)

# Popularity rank of each target book, using the Bayesian score
popularity_ranking = (
    scored_books.merge(book_metadata[["book_id", "title"]], on="book_id", how="inner")
    .sort_values(["bayesian_score", "rating_count"], ascending=[False, False])
    .reset_index(drop=True)
)
popularity_ranking["rank"] = popularity_ranking.index + 1

display(
    popularity_ranking[popularity_ranking["title"].isin(target_titles)]
    .sort_values("rank")[["rank", "title", "rating_count", "bayesian_score"]]
    .round(3)
)


**Analysis: There is an apparent genre match. For example, the Classic Greek mythology (the Odyssey and Iliad) are similar to each other, while the recent Fantasy books Hunger Games and Harry Potter are similarly close. Finally, Pride and Prejudice, the only book that does not have an analogous "twin" of the five, is closer to the modern Fantasy books, but not nearly as close as either grouping. However, popularity matching seems plausible as well, given Harry Potter and Hunger games are the most popular, and the Odyssey and Iliad by far the least.**

4. Write your own IBCF function.

- Input: newuser, a vector of ratings for every book in your catalog. Many entries will be NA; non-NA values are stars in {1, 2, 3, 4, 5}.
- Output: the top 10 book recommendations for this user, as titles. If fewer than 10 predictions are non-NA, fill the remaining slots from the popularity ranking in (A), skipping books the user has already rated. You may want to store that ranking once so you do not recompute it.
- The function should load (or receive) the truncated similarity matrix and predict ratings for unrated books with the weighted-average formula from the notes. State how you handle a book whose neighbors were also unrated by this user.

In [ ]:
popularity_order = (
    scored_books.merge(book_metadata, on="book_id", how="inner")
    .sort_values(["bayesian_score", "rating_count"], ascending=[False, False])
)['title'].tolist()

In [ ]:
def ibcf(newuser, S, titles, popularity_order):
    rated = ~np.isnan(newuser)
    sd = S.iloc[:,rated]
    pred = (sd * newuser[rated]).sum(axis=1) / sd.sum(axis=1).replace(0,np.nan)
    pred[rated] = np.nan
    pred_top10 = pred.dropna().nlargest(10)
    
    recs_books_top10 = pred_top10.index.map(titles.set_index('book_id')['title']).tolist()
    read_books = (titles[rated]['title']).tolist()

    i=0
    while len(recs_books_top10) < 10 and i < len(popularity_order):
        popular_book = popularity_order[i]
        if popular_book not in recs_books_top10 and popular_book not in read_books:
            recs_books_top10.append(popular_book)
        i+=1
            
    return recs_books_top10, pred_top10.tolist()

def predict_ratings(newuser, S):
    rated = ~np.isnan(newuser)
    sd = S.loc[:, rated]
    pred = (sd * newuser[rated]).sum(axis=1) / sd.sum(axis=1).replace(0, np.nan)
    pred[rated] = np.nan
    
    return pred

def user_lookup(uid, rating_list):
    u_ratings = rating_list[rating_list["user_id"] == uid]
    #u_book_ids = u_ratings['book_id'].tolist()
    #user2_book_names = book_metadata.set_index('book_id')['title'].loc[user2_book_ids]

    u_mapping = u_ratings.set_index('book_id')['rating']
    u_lookup = np.array([u_mapping.get(book_id, np.nan) for book_id in book_ids])
    return u_lookup
    #user2_recs_books, user2_recs_scores = ibcf(newuser_2, similarity_df, book_metadata, popularity_order)


**For each book we want to score, we use its retained similar books that the user has rated to compute a similarity-weighted prediction. The prediction will be missing when none of the similar books have been rated by the user.**

5. Test the function by displaying the top 10 recommendations, with predicted scores when you have them, for:

- One real user from your filtered matrix who has between 30 and 80 ratings. Report the user id and a short sketch of what they have already rated (a handful of titles is enough).
- A hypothetical user who rates “Harry Potter and the Sorcerer’s Stone (Harry Potter, #1)” with 5 stars and “Pride and Prejudice” with 5 stars, and nothing else.
- One hypothetical user per group member. Each member picks their favorite book, rates it with 5 stars, and rates nothing else. The title must appear in your filtered catalog (one of the N most-rated books). If someone’s actual favorite is not on that list, they pick the closest book that is, and say so. Report each member’s name, the exact title as it appears in books.csv, and that member’s top 10 list.

  Compare each list, briefly, to the top 10 from (A). How much overlap is there, and what does that say about popularity bias in IBCF?

In [ ]:
from itertools import zip_longest

user2_ratings = ratings_filtered[ratings_filtered["user_id"] == 2]
user2_book_ids = user2_ratings['book_id'].tolist()
user2_book_names = book_metadata.set_index('book_id')['title'].loc[user2_book_ids]

user2_mapping = user2_ratings.set_index('book_id')['rating']
newuser_2 = np.array([user2_mapping.get(book_id, np.nan) for book_id in book_ids])
user2_recs_books, user2_recs_scores = ibcf(newuser_2, similarity_df, book_metadata, popularity_order)

print("User 2 Has Read:")
print(user2_book_names.head(5))
print('\n')
print("User 2 Recommendations:")
for title, score in zip_longest(user2_recs_books, user2_recs_scores, fillvalue=None):
    if user2_recs_scores is None:
        print("Title:", title)
    else:
        print("Title:", title, "\t","Score:", score)

The user with *user_id* 2 is chosen. Some of the books that this user has rated include *How to Win Friends and Influence People*, *The Da Vinci Code (Robert Langdon, #2)*, *Who Moved My Cheese?*, *Memoirs of a Geisha*, and *Heart of Darkness*.

In [ ]:
targets = book_metadata[book_metadata["title"].isin(["Harry Potter and the Sorcerer's Stone (Harry Potter, #1)", "Pride and Prejudice"])]
hypothetical_user_book_ids = targets["book_id"].tolist()
hypothetical_user_mapping = {book_id: 5 for book_id in hypothetical_user_book_ids}
newhypothetical_user = np.array([hypothetical_user_mapping.get(book_id, np.nan) for book_id in book_ids])
hypothetical_user_recs_books, hypothetical_user_recs_scores = ibcf(newhypothetical_user, similarity_df, book_metadata, popularity_order)
print("Hypothetical User Recommendations:")
for title, score in zip_longest(hypothetical_user_recs_books, hypothetical_user_recs_scores, fillvalue=None):
    if hypothetical_user_recs_scores is None:
        print("Title:", title)
    else:
        print("Title:", title, "\t","Score:", score)

In [ ]:
member1_favorite_book = "The Alchemist"
targets = book_metadata[book_metadata["title"].isin([member1_favorite_book])]
member1_book_ids = targets["book_id"].tolist()
member1_book_ids_mapping = {book_id: 5 for book_id in member1_book_ids}
newmember1 = np.array([member1_book_ids_mapping.get(book_id, np.nan) for book_id in book_ids])
newmember1_recs_books, newmember1_recs_scores = ibcf(newmember1, similarity_df, book_metadata, popularity_order)
print("Member1 Favorite Book:", member1_favorite_book)
print("Member1 Recommendations:")
for title, score in zip_longest(newmember1_recs_books, newmember1_recs_scores, fillvalue=None):
    if newmember1_recs_scores is None:
        print("Title:", title)
    else:
        print("Title:", title, "\t","Score:", score)

In [ ]:
member2_favorite_book = "Fifty Shades of Grey (Fifty Shades, #1)"
targets = book_metadata[book_metadata["title"].isin([member2_favorite_book])]
member2_book_ids = targets["book_id"].tolist()
member2_book_ids_mapping = {book_id: 5 for book_id in member2_book_ids}
newmember2 = np.array([member2_book_ids_mapping.get(book_id, np.nan) for book_id in book_ids])
newmember2_recs_books, newmember2_recs_scores = ibcf(newmember2, similarity_df, book_metadata, popularity_order)
print("Member2 Favorite Book:", member2_favorite_book)
print("Member2 Recommendations:")
for title, score in zip_longest(newmember2_recs_books, newmember2_recs_scores, fillvalue=None):
    if newmember2_recs_scores is None:
        print("Title:", title)
    else:
        print("Title:", title, "\t","Score:", score)

In [ ]:
member3_favorite_book = "Hamlet"
targets = book_metadata[book_metadata["title"].isin([member3_favorite_book])]
member3_book_ids = targets["book_id"].tolist()
member3_book_ids_mapping = {book_id: 5 for book_id in member3_book_ids}
newmember3 = np.array([member3_book_ids_mapping.get(book_id, np.nan) for book_id in book_ids])
newmember3_recs_books, newmember3_recs_scores = ibcf(newmember3, similarity_df, book_metadata, popularity_order)
print("Member3 Favorite Book:", member3_favorite_book)
print("Member3 Recommendations:")
for title, score in zip_longest(newmember3_recs_books, newmember3_recs_scores, fillvalue=None):
    if newmember3_recs_scores is None:
        print("Title:", title)
    else:
        print("Title:", title, "\t","Score:", score)

In [ ]:
member4_favorite_book = "The Martian"
targets = book_metadata[book_metadata["title"].isin([member4_favorite_book])]
member4_book_ids = targets["book_id"].tolist()
member4_book_ids_mapping = {book_id: 5 for book_id in member4_book_ids}
newmember4 = np.array([member4_book_ids_mapping.get(book_id, np.nan) for book_id in book_ids])
newmember4_recs_books, newmember4_recs_scores = ibcf(newmember4, similarity_df, book_metadata, popularity_order)
print("Member4 Favorite Book:", member4_favorite_book)
print("Member4 Recommendations:")
for title, score in zip_longest(newmember4_recs_books, newmember4_recs_scores, fillvalue=None):
    if newmember4_recs_scores is None:
        print("Title:", title)
    else:
        print("Title:", title, "\t","Score:", score)

**The hypothetical user’s list and Member4's list both have a two titles that are in the top 10 from (A). This demonstrates the popularity bias in our IBCF implementation where popular items are recommended to users if the user has fewer ratings. If we were to provide more ratings for our members, we expect fewer books would be in the top-ten book list.**

## (C) Evaluation

The lists in (B) show what IBCF recommends. They do not show whether
it is more accurate than a simple baseline. Parts (C1)–(C4) use the same filtered catalog
as (A)–(B), but a different rating matrix.

1. Split ratings, not users. Start from the filtered matrix R of part (B). Among users who have at least 10 observed ratings, randomly select 20% of that user’s observed entries (at least one) and move them to a test set. Replace those test entries in R by NA. Call the result Rtrain. The test set is a list of triples (u, j, ruj) that you no longer treat as observed.

    Because ratings.csv has no per-user timestamp, drawing the test ratings uniformly at random within each user is the default. If you instead cut the file using its global time order, say so and defend the choice.

In [ ]:
MIN_RATINGS = 10
RANDOM_SEED = 598

# get users with >= 10 ratings
#test_users = np.flatnonzero(ratings_per_user >= MIN_RATINGS)
test_user_ratings_count = ratings_filtered["user_id"].value_counts()
test_users = (test_user_ratings_count[test_user_ratings_count >= MIN_RATINGS].index).sort_values()
test_ratings = ratings_filtered[ratings_filtered['user_id'].isin(test_users)]

# randomly select 20% of each user's entries, move to test set
#rng = np.random.default_rng(RANDOM_SEED)
#test_rows, test_cols = [], []

test_set = (
    test_ratings.groupby('user_id', group_keys=False)
    .sample(frac=0.2, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

###test_set = pd.DataFrame(columns=["user_id", "book_id", "rating"])
###for u in test_users:
###    #alternative based on ratings?
###    u_ratings = ratings_filtered[ratings_filtered['user_id'] == u].sample(frac=0.2, random_state=RANDOM_SEED)
###
###    test_set = pd.concat([test_set, u_ratings], ignore_index = True)
###    #rated_books = R[u].indices
###    #n_test = max(1, round(0.2 * len(rated_books)))
###    #picked = rng.choice(rated_books, size=n_test, replace=False)
###    #test_rows.extend([u] * n_test)
###    #test_cols.extend(picked)
    
# save test set
#test_set = pd.DataFrame({
#    "user_id": test_rows,
#    "book_id": test_cols,
#    "rating": np.asarray(R[test_rows, test_cols]).ravel(),
#})

# Create R_train, replace test entries in R_train by NA
#R_train = R.copy()
#for index, row in test_set.iterrows():
#    R_train[row["user_id"], row["book_id"]] = 0

#R_train[test_rows, test_cols] = 0
#R_train.eliminate_zeros()

#book_ids = books_filtered["book_id"].to_list()
#book_index = {book_id: i for i, book_id in enumerate(book_ids)}
#test_user_codes, test_unique_users = pd.factorize(test_set["user_id"])
#test_item_codes = test_set["book_id"].map(book_index).to_numpy()

train_ratings = ratings_filtered.merge(test_set, on=['user_id', 'book_id', 'rating'], how='left', indicator=True)
train_ratings = train_ratings[train_ratings['_merge'] == 'left_only'].drop(columns=['_merge'])

train_user_codes, train_unique_users = pd.factorize(train_ratings["user_id"])
train_item_codes = train_ratings["book_id"].map(book_index).to_numpy()

R_train = csr_matrix(
    (train_ratings["rating"].to_numpy(dtype=float), (train_user_codes, train_item_codes)),
    shape=(len(train_unique_users), len(book_ids)),
)
R_train.sort_indices()

print("Test ratings:", len(test_set))
print("Train ratings:", R_train.nnz)

**How the test ratings were drawn and why.**

**How** Test ratings were drawn from the filtered rating matrix R, from part B, from users with at least 10 observed ratings. Of those users, 20% of their ratings were selected **uniformly at random** (with seed 598) and copied to a test set.

**Why** The dataset, ratings.csv, is sorted by time, so it is possible to cut the file by global time order, for example using the last 20% of the dataset as a test set, and using the first 80% as a training set. However, taking this approach would make ratings from users land unevenly in test and train datasets. This is not optimal for our goal, since we want to predict the rating a user would give a new book from the ratings they've already given. For example, if we split by global time order, a user whose ratings fall mostly near the end of the file would have few or no ratings left in training, so IBCF would have nothing of theirs to predict from and would fall back to the popularity ranking. So, we instead chose to split each user's ratings uniformly at random, given they had at least 10 ratings. Therefore IBCF always has at least 8 of their own ratings to predict from, while at least 2 are held out to test those predictions.

2. Refit IBCF on Rtrain only. The similarity matrix S from part (B) was built using every observed rating, including the ones that are now test data. Do not use that S here. On Rtrain — that is, after the test entries have been set to NA — recompute, in order:

- the row means (from the remaining non-NA entries only),
- the centered cosine similarities,
- the rule that drops pairs with fewer than 3 co-raters,
- the top-30 truncation.

    Then pass Rtrain and this new S into the IBCF function from (B)4. If one object S in
your code is used both for the lists in (B) and for the numbers in (C), the test ratings
have leaked into the neighbor graph. Write one sentence in the report confirming
that S was recomputed from Rtrain.

In [ ]:
# recompute row means from training ratings
ratings_per_user_train = np.diff(R_train.indptr)
user_means_train = np.divide(
    np.asarray(R_train.sum(axis=1)).ravel(),
    ratings_per_user_train,
    out=np.zeros(R_train.shape[0]),
    where=ratings_per_user_train > 0,
)

# center each row by new mean
R_train_centered = R_train.copy()
R_train_centered.data -= np.repeat(user_means_train, ratings_per_user_train)

# cosine similarity, transformed to nonnegative weights
similarities_train = (1 + cosine_similarity(R_train_centered.T)) / 2

# count how many users rated each pair of books in R_train
rated_train = R_train.copy()
rated_train.data[:] = 1
co_raters_train = (rated_train.T @ rated_train).toarray()

# missing pairs with fewer than 3 co-raters and 0 length book vectors
book_norms_train = np.sqrt(np.asarray(R_train_centered.power(2).sum(axis=0)).ravel())
similarities_train[co_raters_train < 3] = np.nan
similarities_train[:, book_norms_train == 0] = np.nan
similarities_train[book_norms_train == 0, :] = np.nan
np.fill_diagonal(similarities_train, np.nan)

# keep only top 30 similarities in each row
top_neighbors_train = np.full_like(similarities_train, np.nan)
for i, row in enumerate(similarities_train):
    candidates = np.flatnonzero(np.isfinite(row))
    keep = candidates[np.argsort(row[candidates])[-k:]]
    top_neighbors_train[i, keep] = row[keep]

similarity_df_train = pd.DataFrame(top_neighbors_train, index=book_ids, columns=book_ids)
print(pd.DataFrame(similarities_train[:10, :10]).round(2))

**The similarity matrix S was recomputed from Rtrain:** the row means, centering, cosine similarities, co-rater counts, and top-30 truncation were all computed from Rtrain after the test entries were set to NA.

3. Rating error. Using only the held-out triples from (C)1, report RMSE (MAE optional) for at least three predictors:
- a popularity or global-mean baseline consistent with part (A),
- a user-mean baseline (or μ + ai + bj , if you implement it),
- IBCF, with the S fit on Rtrain.

    If IBCF cannot predict a test pair (u, j) because u has no rated neighbor of j in
Rtrain, fill that prediction with the popularity fallback from (B)4. Do not drop the
pair. Report how many test pairs used the fallback. 

Popularity Baseline

In [ ]:
print(len(all_m500))
print(len(filtered_all_m500))

In [ ]:
from sklearn.metrics import root_mean_squared_error

# Fit the Bayesian popularity scores using training ratings only.
m = 500
training_global_mean = train_ratings["rating"].mean()
train_book_stats = (
    train_ratings.groupby("book_id")["rating"]
    .agg(rating_count="count", mean_rating="mean")
)
train_book_stats["bayesian_score"] = (
    train_book_stats["rating_count"] * train_book_stats["mean_rating"]
    + m * training_global_mean
) / (train_book_stats["rating_count"] + m)
train_popularity_score_by_book = train_book_stats["bayesian_score"].to_dict()

# Use test_set only for ground-truth ratings when evaluating.
popularity_predictions = (
    test_set["book_id"]
    .map(train_popularity_score_by_book)
    .fillna(training_global_mean)
)
popularity_rmse = root_mean_squared_error(test_set["rating"], popularity_predictions)
print(f"Popularity RMSE: {popularity_rmse}")

User-Mean Baseline

In [ ]:
#TODO Mark this isn't valid - calculate dfrom held-out and then compared to those same ratings.
# 
# test_user_means = test_set.groupby("user_id")["rating"].mean()

user_mean_rmse = root_mean_squared_error(user_means[test_users - 1], test_user_means)

print(f"User-Mean RMSE: {user_mean_rmse}")


IBCF, with the S fit on Rtrain

In [ ]:
print(scored_books)

In [ ]:
# If IBCF cannot predict a test pair, use the training-only popularity score as fallback.
preds = []
nan_count = 0
lastid = None

for urow in test_set.itertuples(index=False):
    if urow.user_id != lastid:
        lastid = urow.user_id
        ulookup = user_lookup(lastid, train_ratings)
        predictions = predict_ratings(ulookup, similarity_df_train)

    pred_book = predictions[urow.book_id]
    if pd.isna(pred_book):
        nan_count += 1
        pred_book = train_popularity_score_by_book.get(urow.book_id, training_global_mean)
    preds.append(pred_book)

print(f"Test pairs using popularity fallback: {nan_count}")

ibcf_rmse = root_mean_squared_error(test_set["rating"], preds)
print(f"IBCF RMSE: {ibcf_rmse}")

4. Ranking error. RMSE scores a predicted number of stars. A recommender returns a list. Also report Precision@10 (Recall@10 or NDCG@10 optional). For each test user, take the 10 books IBCF ranks highest among books that user did not rate in Rtrain. A book is relevant if that user’s held-out rating for it is 4 or 5. Optional: treat books on that user’s to read list as additional relevant items and say whether Precision@10 changes.

In [ ]:
for u in test_users[:10]:
    print(u)
    testuser_ratings = test_set[test_set["user_id"] == u]
    testuser_book_ids = testuser_ratings['book_id'].tolist()
    testuser_book_names = book_metadata.set_index('book_id')['title'].loc[testuser_book_ids]

    testuser_mapping = testuser_ratings.set_index('book_id')['rating']
    testuser = np.array([testuser_mapping.get(book_id, np.nan) for book_id in book_ids])
    testuser_recs_books, testuser_recs_scores = ibcf(testuser, similarity_df_train, book_metadata, popularity_order)
    print(testuser_recs_books)
    print(testuser_recs_scores)

5. In a short paragraph: did IBCF beat the baselines on RMSE? On Precision@10? If the two metrics disagree, which one would you trust for a “top 10 books for you” product, and why? TODO MARK

TODO
**Paragraph**